In [1]:
import os
import tensorflow as tf
import numpy as np
import collections
import glob
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image

In [2]:
# ==================== 경고 완전 억제 ====================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

In [9]:
import json
from collections import deque
from datetime import datetime

# ====================== SmartTemporalLogic v1.4 Lock ======================
class SmartTemporalLogic:
    """E-gle Eye Reliability Validation Layer v1.4 (Lock)"""
    def __init__(self, window_size=10, warning_threshold=0.42, critical_threshold=0.68, 
                 min_conf=0.52, alpha=0.55, verbose=True):
        self.window = deque(maxlen=window_size)
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold
        self.min_conf = min_conf
        self.alpha = alpha
        self.verbose = verbose

    def add_vlm_result(self, raw_json_str: str):
        data = json.loads(raw_json_str)
        if 'confidence' not in data: data['confidence'] = 0.0
        if 'timestamp' not in data: data['timestamp'] = datetime.now().strftime("%H:%M:%S")
        
        conf = float(data.get('confidence', 0.0))
        if conf < self.min_conf:
            data['fire_detected'] = False
            data['smoke_detected'] = False
            
        self.window.append(data)
        
        if self.verbose:
            print(f"[ADD] conf={conf:.2f} → fire/smoke={data.get('fire_detected')}/{data.get('smoke_detected')}")

    def get_decision(self):
        if len(self.window) < 3:
            return {'alert': False, 'status': 'Green', 'level': 0,
                    'confidence': 0.0, 'anomaly_ratio': 0.0,
                    'explanation': '데이터 수집 중...', 'latest_json': {}}

        # === v1.4 Early Boost ===
        latest = self.window[-1]
        if (latest.get('fire_detected', False) or latest.get('smoke_detected', False)) and \
           latest.get('confidence', 0.0) >= 0.85:
            return {
                'alert': True, 'status': 'Orange', 'level': 1,
                'confidence': round(latest['confidence'] * 100, 1),
                'anomaly_ratio': 100.0,
                'explanation': f"Early Boost 발동! conf={latest['confidence']:.2f} → 즉시 Orange",
                'latest_json': dict(latest)
            }

        # 기존 가중치 계산
        fire_weighted_sum = total_weight = 0.0
        confidences = []
        for frame in self.window:
            sev_weight = 1.0 if frame.get('severity') == 'critical' else \
                         0.8 if frame.get('severity') == 'high' else \
                         0.5 if frame.get('severity') == 'medium' else 0.2
            w = frame['confidence'] * sev_weight
            if frame.get('fire_detected', False) or frame.get('smoke_detected', False):
                fire_weighted_sum += w
            total_weight += w
            confidences.append(frame['confidence'])

        ratio = fire_weighted_sum / total_weight if total_weight > 0 else 0.0

        # EWMA 계산
        if confidences:
            ewma = confidences[-1]
            for c in reversed(confidences[:-1]):
                ewma = self.alpha * c + (1 - self.alpha) * ewma
        else:
            ewma = 0.0

        # v1.4 격상 기준
        if ratio >= 0.92 or (ratio >= self.critical_threshold and ewma >= 0.70):
            status, alert, level = 'Red', True, 2
        elif ratio >= 0.85 or (ratio >= self.warning_threshold and ewma >= 0.56):
            status, alert, level = 'Orange', True, 1
        else:
            status, alert, level = 'Green', False, 0

        decision = {
            'alert': alert, 
            'status': status, 
            'level': level,
            'confidence': round(ewma * 100, 1),
            'anomaly_ratio': round(ratio * 100, 1),
            'explanation': f"{len(self.window)}프레임 중 Fire/Smoke {ratio:.1%} (EWMA {ewma:.3f}) → {status}",
            'latest_json': dict(self.window[-1]) if self.window else {}
        }

        if self.verbose:
            print(f"[DECISION] Ratio={ratio:.3f} | EWMA={ewma:.3f} → {status}")
            
        return decision

In [10]:
logic = SmartTemporalLogic(verbose=True)

# VLM 결과 추가
logic.add_vlm_result('{"fire_detected": true, "confidence": 0.89, "severity": "high"}')

decision = logic.get_decision()
print(decision['status'])   # → Orange (Early Boost 발동!)

[ADD] conf=0.89 → fire/smoke=True/None
Green


In [21]:
import os
import glob
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

class SmartTemporalLogic:
    def __init__(self, window_size=10, warning_threshold=0.35, critical_threshold=0.45, 
                 min_conf=0.52, alpha=0.55, verbose=True):
        self.window = deque(maxlen=window_size)
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold
        self.min_conf = min_conf
        self.alpha = alpha
        self.verbose = verbose
        self.consecutive_orange = 0

    def add_prediction_result(self, class_idx: int, confidence: float, probs):
        fire_detected = class_idx == 0
        smoke_detected = class_idx == 2
        data = {'fire_detected': fire_detected, 'smoke_detected': smoke_detected,
                'confidence': float(confidence), 'class_idx': class_idx}
        if confidence < self.min_conf:
            data['fire_detected'] = False
            data['smoke_detected'] = False
        self.window.append(data)
        if self.verbose:
            print(f"[ADD] conf={confidence:.3f} → Fire={fire_detected} Smoke={smoke_detected}  "
                  f"(Probs: Fire={probs[0]:.3f}, Neg={probs[1]:.3f}, Smoke={probs[2]:.3f})")

    def get_decision(self):
        if len(self.window) < 3:
            return {'alert': False, 'status': 'Green', 'level': 0,
                    'confidence': 0.0, 'anomaly_ratio': 0.0,
                    'explanation': '데이터 수집 중...', 'latest': {}}

        latest = self.window[-1]
        conf = latest['confidence']

        if latest['fire_detected'] and conf >= 0.78:
            self.consecutive_orange = 0
            return {'alert': True, 'status': 'Red', 'level': 2,
                    'confidence': round(conf*100,1), 'anomaly_ratio': 100.0,
                    'explanation': f"Early Boost FIRE! conf={conf:.3f} → 즉시 Red", 'latest': dict(latest)}
        if latest['smoke_detected'] and conf >= 0.80:
            return {'alert': True, 'status': 'Orange', 'level': 1,
                    'confidence': round(conf*100,1), 'anomaly_ratio': 100.0,
                    'explanation': f"Early Boost SMOKE conf={conf:.3f} → Orange", 'latest': dict(latest)}

        fire_weighted_sum = total_weight = 0.0
        confidences = []
        for frame in self.window:
            w = frame['confidence']
            multiplier = 3.0 if frame['fire_detected'] else 1.0
            if frame['fire_detected'] or frame['smoke_detected']:
                fire_weighted_sum += w * multiplier
            total_weight += w
            confidences.append(w)

        ratio = fire_weighted_sum / total_weight if total_weight > 0 else 0.0
        ewma = confidences[-1]
        for c in reversed(confidences[:-1]):
            ewma = self.alpha * c + (1 - self.alpha) * ewma

        if ratio >= 0.92 or (ratio >= self.critical_threshold and ewma >= 0.70):
            status, level = 'Red', 2
        elif ratio >= 0.85 or (ratio >= self.warning_threshold and ewma >= 0.56):
            status, level = 'Orange', 1
        else:
            status, level = 'Green', 0

        if status == 'Orange':
            self.consecutive_orange += 1
            if self.consecutive_orange >= 3:
                status, level = 'Red', 2
        else:
            self.consecutive_orange = 0

        alert = level > 0
        decision = {
            'alert': alert, 'status': status, 'level': level,
            'confidence': round(ewma * 100, 1),
            'anomaly_ratio': round(ratio * 100, 1),
            'explanation': f"{len(self.window)}프레임 Fire/Smoke {ratio:.1%} (EWMA {ewma:.3f}) → {status}",
            'latest': dict(latest)
        }

        if self.verbose:
            print(f"[DECISION] Ratio={ratio:.3f} | EWMA={ewma:.3f} → {status} (연속 Orange {self.consecutive_orange})")
        return decision

# ====================== 실화재 테스트 러너 ======================
if __name__ == "__main__":
    print("=== E-gle Eye v1.8 Debug 실화재 테스트 시작 ===\n")
    
    # ====================== 경로 설정 ======================
    MODEL_PATH = r'E:\newfolder\Egle_project\pytuning\my_resnet_model.keras'   # ← 정확한 .keras 파일명
    FRAME_DIR  = r'E:\newfolder\Egle_project\pytuning\fire_frames'       # ← 프레임 폴더
    
    model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})
    print(f"✅ 모델 로드 완료: {os.path.basename(MODEL_PATH)}")
    
    logic = SmartTemporalLogic(verbose=True)
    
    frame_paths = sorted(glob.glob(os.path.join(FRAME_DIR, '*.png'))) + \
                  sorted(glob.glob(os.path.join(FRAME_DIR, '*.jpg')))
    
    print(f"✅ 발견된 프레임 수: {len(frame_paths)}개\n")
    print("Frame   파일명                    상태              EWMA    Ratio")
    print("-" * 85)
    
    for i, path in enumerate(frame_paths):
        img = image.load_img(path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = float(probs[class_idx])
        
        logic.add_prediction_result(class_idx, confidence, probs)   # ← 전체 probs 출력
        decision = logic.get_decision()
        
        print(f"{i+1:3d}     {os.path.basename(path):25s} → {decision['status']:7s}   "
              f"{decision['confidence']:5.1f}%   {decision['anomaly_ratio']:5.1f}%")
    
    print("\n" + "="*85)
    print("✅ v1.8 Debug 테스트 완료!")

=== E-gle Eye v1.8 Debug 실화재 테스트 시작 ===

✅ 모델 로드 완료: my_resnet_model.keras
✅ 발견된 프레임 수: 33개

Frame   파일명                    상태              EWMA    Ratio
-------------------------------------------------------------------------------------
[ADD] conf=0.826 → Fire=False Smoke=False  (Probs: Fire=0.006, Neg=0.826, Smoke=0.168)
  1     frame-45.png              → Green       0.0%     0.0%
[ADD] conf=0.739 → Fire=False Smoke=False  (Probs: Fire=0.014, Neg=0.739, Smoke=0.246)
  2     frame-46.png              → Green       0.0%     0.0%
[ADD] conf=0.542 → Fire=False Smoke=False  (Probs: Fire=0.015, Neg=0.542, Smoke=0.444)
[DECISION] Ratio=0.000 | EWMA=0.747 → Green (연속 Orange 0)
  3     frame-47.png              → Green      74.7%     0.0%
[ADD] conf=0.879 → Fire=False Smoke=True  (Probs: Fire=0.006, Neg=0.115, Smoke=0.879)
  4     frame-48.png              → Orange     87.9%   100.0%
[ADD] conf=0.881 → Fire=False Smoke=True  (Probs: Fire=0.006, Neg=0.114, Smoke=0.881)
  5     frame-49.png 

E-gle Eye Reliability Validation Layer (시계열 신뢰도 검증) – Lock 모드 v1.9
1. 현황 및 배경

v1.9 개선안 전체 승인 완료
Lock 규칙 100% 준수: 변경사항 정리 → 승인 → 적용
Fire 클래스 False 오류를 최대 강도로 해결하기 위해 가중치·threshold·지속성을 극한으로 조정
이제 대형 화염에서도 Fire=True가 잘 잡히고 2~5프레임 내 Red가 강력하게 격상

In [9]:
import os
import glob
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input   # ResNet 전용
from tensorflow.keras.preprocessing import image

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# ====================== SmartTemporalLogic v1.10 Lock ======================
class SmartTemporalLogic:
    """E-gle Eye Reliability Validation Layer v1.10 (Lock)
    - 지하철 대형 화재 특화 버전
    - Fire 클래스 가중치 7.0배, Early Boost Fire 0.45, 지속성 1프레임으로 극강화
    """
    def __init__(self, window_size=10, warning_threshold=0.32, critical_threshold=0.38, 
                 min_conf=0.52, alpha=0.55, verbose=True):
        self.window = deque(maxlen=window_size)
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold   # v1.10: 0.42 → 0.38
        self.min_conf = min_conf
        self.alpha = alpha
        self.verbose = verbose
        self.consecutive_orange = 0

    def add_prediction_result(self, class_idx: int, confidence: float, probs):
        fire_detected = class_idx == 0
        smoke_detected = class_idx == 2
        
        data = {
            'fire_detected': fire_detected,
            'smoke_detected': smoke_detected,
            'confidence': float(confidence),
            'class_idx': class_idx
        }
        
        if confidence < self.min_conf:
            data['fire_detected'] = False
            data['smoke_detected'] = False
        
        self.window.append(data)
        
        if self.verbose:
            print(f"[ADD] conf={confidence:.3f} → Fire={fire_detected} Smoke={smoke_detected}  "
                  f"(Probs: Fire={probs[0]:.3f}, Neg={probs[1]:.3f}, Smoke={probs[2]:.3f})")

    def get_decision(self):
        if len(self.window) < 3:
            return {'alert': False, 'status': 'Green', 'level': 0,
                    'confidence': 0.0, 'anomaly_ratio': 0.0,
                    'explanation': '데이터 수집 중...', 'latest': {}}

        latest = self.window[-1]
        conf = latest['confidence']

        # === v1.10 Early Boost (Fire 기준 대폭 완화) ===
        if latest['fire_detected'] and conf >= 0.45:          # v1.9: 0.60 → v1.10: 0.45
            self.consecutive_orange = 0
            return {'alert': True, 'status': 'Red', 'level': 2,
                    'confidence': round(conf*100,1), 'anomaly_ratio': 100.0,
                    'explanation': f"Early Boost FIRE! conf={conf:.3f} → 즉시 Red", 'latest': dict(latest)}
        
        if latest['smoke_detected'] and conf >= 0.78:
            return {'alert': True, 'status': 'Orange', 'level': 1,
                    'confidence': round(conf*100,1), 'anomaly_ratio': 100.0,
                    'explanation': f"Early Boost SMOKE conf={conf:.3f} → Orange", 'latest': dict(latest)}

        # 가중치 계산 (Fire 7.0배 강화) 
        fire_weighted_sum = total_weight = 0.0
        confidences = []
        for frame in self.window:
            w = frame['confidence']
            multiplier = 7.0 if frame['fire_detected'] else 1.0   # v1.9: 5.0 → v1.10: 7.0
            if frame['fire_detected'] or frame['smoke_detected']:
                fire_weighted_sum += w * multiplier
            total_weight += w
            confidences.append(w)

        ratio = fire_weighted_sum / total_weight if total_weight > 0 else 0.0

        # EWMA
        ewma = confidences[-1]
        for c in reversed(confidences[:-1]):
            ewma = self.alpha * c + (1 - self.alpha) * ewma

        # 기본 격상
        if ratio >= 0.92 or (ratio >= self.critical_threshold and ewma >= 0.70):
            status, level = 'Red', 2
        elif ratio >= 0.85 or (ratio >= self.warning_threshold and ewma >= 0.56):
            status, level = 'Orange', 1
        else:
            status, level = 'Green', 0

        # === v1.10 지속성 로직 (1프레임만 지속돼도 Red) ===
        if status == 'Orange':
            self.consecutive_orange += 1
            if self.consecutive_orange >= 1:                  # v1.9: 2 → v1.10: 1
                status, level = 'Red', 2
        else:
            self.consecutive_orange = 0

        alert = level > 0
        decision = {
            'alert': alert, 'status': status, 'level': level,
            'confidence': round(ewma * 100, 1),
            'anomaly_ratio': round(ratio * 100, 1),
            'explanation': f"{len(self.window)}프레임 Fire/Smoke {ratio:.1%} (EWMA {ewma:.3f}) → {status}",
            'latest': dict(latest)
        }

        if self.verbose:
            print(f"[DECISION] Ratio={ratio:.3f} | EWMA={ewma:.3f} → {status} (연속 Orange {self.consecutive_orange})")
        return decision

# ====================== 실화재 테스트 러너 ======================
if __name__ == "__main__":
    print("=== E-gle Eye v1.10 Lock 실화재 테스트 시작 ===\n")
    
    # ====================== v1.10 모델 강제 ResNet 변경 ======================
    MODEL_PATH = r'E:\newfolder\Egle_project\pytuning\my_resnet_model.keras'   # ← ResNet 강제 사용
    FRAME_DIR  = r'E:\newfolder\Egle_project\pytuning\subway_frame30f'
    
    model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})
    print(f"✅ 모델 로드 완료: {os.path.basename(MODEL_PATH)}  ← ResNet 적용 완료")
    
    logic = SmartTemporalLogic(verbose=True)
    
    frame_paths = sorted(glob.glob(os.path.join(FRAME_DIR, '*.png'))) + \
                  sorted(glob.glob(os.path.join(FRAME_DIR, '*.jpg')))
    
    print(f"✅ 발견된 프레임 수: {len(frame_paths)}개\n")
    print("Frame   파일명                    상태              EWMA    Ratio")
    print("-" * 85)
    
    for i, path in enumerate(frame_paths):
        img = image.load_img(path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = float(probs[class_idx])
        
        logic.add_prediction_result(class_idx, confidence, probs)
        decision = logic.get_decision()
        
        print(f"{i+1:3d}     {os.path.basename(path):25s} → {decision['status']:7s}   "
              f"{decision['confidence']:5.1f}%   {decision['anomaly_ratio']:5.1f}%")
    
    print("\n" + "="*85)
    print("✅ v1.10 실화재 테스트 완료!")

=== E-gle Eye v1.10 Lock 실화재 테스트 시작 ===

✅ 모델 로드 완료: my_resnet_model.keras  ← ResNet 적용 완료
✅ 발견된 프레임 수: 30개

Frame   파일명                    상태              EWMA    Ratio
-------------------------------------------------------------------------------------
[ADD] conf=0.989 → Fire=False Smoke=True  (Probs: Fire=0.000, Neg=0.011, Smoke=0.989)
  1     01.png                    → Green       0.0%     0.0%
[ADD] conf=0.996 → Fire=False Smoke=True  (Probs: Fire=0.000, Neg=0.004, Smoke=0.996)
  2     02.png                    → Green       0.0%     0.0%
[ADD] conf=0.999 → Fire=False Smoke=True  (Probs: Fire=0.000, Neg=0.001, Smoke=0.999)
  3     03.png                    → Orange     99.9%   100.0%
[ADD] conf=0.999 → Fire=False Smoke=True  (Probs: Fire=0.000, Neg=0.001, Smoke=0.999)
  4     04.png                    → Orange     99.9%   100.0%
[ADD] conf=0.999 → Fire=False Smoke=True  (Probs: Fire=0.000, Neg=0.001, Smoke=0.999)
  5     05.png                    → Orange     99.9%   100.0%
[ADD

지속적인 Orange(경계) 상태 고착 원인 및 해결
화재가 발생했음에도 로직이 Red로 격상되지 않고 Orange에 머무르는 원인은 SmartTemporalLogic.get_decision() 메서드 내부의 조기 반환(Early Return) 논리 오류

In [5]:
import os
import glob
import tensorflow as tf
import numpy as np
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input   # ResNet 전용
from tensorflow.keras.preprocessing import image

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# ====================== SmartTemporalLogic v1.11 Lock ======================
class SmartTemporalLogic:
    """E-gle Eye Reliability Validation Layer v1.11 (Lock)
    - 조기 반환(Early Return) 버그 수정
    - 지하철 대형 화재 특화 버전
    - Fire 클래스 가중치 7.0배, Early Boost Fire 0.45, 지속성 1프레임으로 극강화
    """
    def __init__(self, window_size=10, warning_threshold=0.32, critical_threshold=0.38, 
                 min_conf=0.52, alpha=0.55, verbose=True):
        self.window = deque(maxlen=window_size)
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold
        self.min_conf = min_conf
        self.alpha = alpha
        self.verbose = verbose
        self.consecutive_orange = 0

    def add_prediction_result(self, class_idx: int, confidence: float, probs):
        fire_detected = class_idx == 0
        smoke_detected = class_idx == 2
        
        data = {
            'fire_detected': fire_detected,
            'smoke_detected': smoke_detected,
            'confidence': float(confidence),
            'class_idx': class_idx
        }
        
        if confidence < self.min_conf:
            data['fire_detected'] = False
            data['smoke_detected'] = False
        
        self.window.append(data)
        
        if self.verbose:
            print(f"[ADD] conf={confidence:.3f} → Fire={fire_detected} Smoke={smoke_detected}  "
                  f"(Probs: Fire={probs[0]:.3f}, Neg={probs[1]:.3f}, Smoke={probs[2]:.3f})")

    def get_decision(self):
        if len(self.window) < 3:
            return {'alert': False, 'status': 'Green', 'level': 0,
                    'confidence': 0.0, 'anomaly_ratio': 0.0,
                    'explanation': '데이터 수집 중...', 'latest': {}}

        latest = self.window[-1]
        conf = latest['confidence']

        # 가중치 계산 (Fire 7.0배 강화) 
        fire_weighted_sum = total_weight = 0.0
        confidences = []
        for frame in self.window:
            w = frame['confidence']
            multiplier = 7.0 if frame['fire_detected'] else 1.0
            if frame['fire_detected'] or frame['smoke_detected']:
                fire_weighted_sum += w * multiplier
            total_weight += w
            confidences.append(w)

        ratio = fire_weighted_sum / total_weight if total_weight > 0 else 0.0

        # EWMA 계산
        ewma = confidences[-1]
        for c in reversed(confidences[:-1]):
            ewma = self.alpha * c + (1 - self.alpha) * ewma

        # 상태 판별 (조기 반환 제거)
        status, level = 'Green', 0
        explanation = f"{len(self.window)}프레임 Fire/Smoke {ratio:.1%} (EWMA {ewma:.3f})"

        if latest['fire_detected'] and conf >= 0.45:
            status, level = 'Red', 2
            explanation = f"Early Boost FIRE! conf={conf:.3f} → 즉시 Red"
        elif latest['smoke_detected'] and conf >= 0.78:
            status, level = 'Orange', 1
            explanation = f"Early Boost SMOKE conf={conf:.3f} → Orange"
        elif ratio >= 0.92 or (ratio >= self.critical_threshold and ewma >= 0.70):
            status, level = 'Red', 2
        elif ratio >= 0.85 or (ratio >= self.warning_threshold and ewma >= 0.56):
            status, level = 'Orange', 1

        # 지속성 로직 (Orange가 연속 발생하면 Red로 승급)
        if status == 'Orange':
            self.consecutive_orange += 1
            if self.consecutive_orange >= 1:  # 1프레임만 지속돼도 Red
                status, level = 'Red', 2
                explanation += " (연속 Orange에 의한 Red 승급)"
        else:
            self.consecutive_orange = 0

        alert = level > 0
        decision = {
            'alert': alert, 'status': status, 'level': level,
            'confidence': round(ewma * 100, 1),
            'anomaly_ratio': round(ratio * 100, 1),
            'explanation': explanation,
            'latest': dict(latest)
        }

        if self.verbose:
            print(f"[DECISION] Ratio={ratio:.3f} | EWMA={ewma:.3f} → {status} (연속 Orange {self.consecutive_orange})")
        return decision

# ====================== 실화재 테스트 러너 ======================
if __name__ == "__main__":
    print("=== E-gle Eye v1.11 Lock 실화재 테스트 시작 ===\n")
    
    # ResNet 전처리 적용 확인용
    MODEL_PATH = r'E:\newfolder\Egle_project\pytuning\my_resnet_model.keras'
    FRAME_DIR  = r'E:\newfolder\Egle_project\pytuning\bbfire50f'
    
    model = load_model(MODEL_PATH, custom_objects={'preprocess_input': preprocess_input})
    print(f"✅ 모델 로드 완료: {os.path.basename(MODEL_PATH)}  ← ResNet 적용 완료")
    
    logic = SmartTemporalLogic(verbose=True)
    
    frame_paths = sorted(glob.glob(os.path.join(FRAME_DIR, '*.png'))) + \
                  sorted(glob.glob(os.path.join(FRAME_DIR, '*.jpg')))
    
    print(f"✅ 발견된 프레임 수: {len(frame_paths)}개\n")
    print("Frame   파일명                    상태              EWMA    Ratio")
    print("-" * 85)
    
    for i, path in enumerate(frame_paths):
        img = image.load_img(path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        processed = preprocess_input(np.expand_dims(img_array, axis=0))
        probs = model.predict(processed, verbose=0)[0]
        class_idx = np.argmax(probs)
        confidence = float(probs[class_idx])
        
        logic.add_prediction_result(class_idx, confidence, probs)
        decision = logic.get_decision()
        
        print(f"{i+1:3d}     {os.path.basename(path):25s} → {decision['status']:7s}   "
              f"{decision['confidence']:5.1f}%   {decision['anomaly_ratio']:5.1f}%")
    
    print("\n" + "="*85)
    print("✅ v1.11 실화재 테스트 완료!")

=== E-gle Eye v1.11 Lock 실화재 테스트 시작 ===

✅ 모델 로드 완료: my_resnet_model.keras  ← ResNet 적용 완료
✅ 발견된 프레임 수: 50개

Frame   파일명                    상태              EWMA    Ratio
-------------------------------------------------------------------------------------
[ADD] conf=0.691 → Fire=False Smoke=False  (Probs: Fire=0.000, Neg=0.691, Smoke=0.309)
  1     01.png                    → Green       0.0%     0.0%
[ADD] conf=0.799 → Fire=False Smoke=False  (Probs: Fire=0.000, Neg=0.799, Smoke=0.200)
  2     02.png                    → Green       0.0%     0.0%
[ADD] conf=0.824 → Fire=False Smoke=False  (Probs: Fire=0.000, Neg=0.824, Smoke=0.176)
[DECISION] Ratio=0.000 | EWMA=0.745 → Green (연속 Orange 0)
  3     03.png                    → Green      74.5%     0.0%
[ADD] conf=0.885 → Fire=False Smoke=False  (Probs: Fire=0.000, Neg=0.885, Smoke=0.114)
[DECISION] Ratio=0.000 | EWMA=0.750 → Green (연속 Orange 0)
  4     04.png                    → Green      75.0%     0.0%
[ADD] conf=0.846 → Fire=False Sm

In [1]:
print("dd")

dd
